In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/fvfvsdfvsfvs/jobs.csv
/kaggle/input/fvfvsdfvsfvs/jobs (1).csv
/kaggle/input/vinai-phobert-base/vinai-phobert-base/config.json
/kaggle/input/vinai-phobert-base/vinai-phobert-base/bpe.codes
/kaggle/input/vinai-phobert-base/vinai-phobert-base/tokenizer_config.json
/kaggle/input/vinai-phobert-base/vinai-phobert-base/model.safetensors
/kaggle/input/vinai-phobert-base/vinai-phobert-base/special_tokens_map.json
/kaggle/input/vinai-phobert-base/vinai-phobert-base/vocab.txt
/kaggle/input/vinai-phobert-base/vinai-phobert-base/added_tokens.json


In [2]:
# PhoBERT Job Pipeline - Kaggle Notebook
# Purpose: Step-by-step data inspection, cleaning, merging, transformation, and baseline model training.
# Saves outputs after each stage to /kaggle/working or ./artifacts so you can resume.

# Stages:
# 0. Setup: imports, path detection
# 1. Inspect datasets
# 2. Cleaning functions for both datasets
# 3. Normalize salary, experience, education, location
# 4. Merge into unified schema and save
# 5. Feature engineering: create input_text and target salary_mean
# 6. Prepare embeddings (SentenceTransformer) and save
# 7. Train baseline model (XGBoost on embeddings) and save
# 8. Save tokenizer & model placeholders for PhoBERT fine-tuning

# NOTES:
# - This notebook is designed to run in a single Kaggle notebook environment.
# - Adjust paths if your CSVs live elsewhere (e.g., /kaggle/input/<dataset>/jobs.csv).
# - We save artifacts after each stage to ./artifacts (and /kaggle/working/artifacts on Kaggle).

# ============================================================================
# Stage 0: Setup
# ============================================================================
import os
import sys
from pathlib import Path
import json
import re
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

ARTIFACT_DIR = Path(os.environ.get('ARTIFACT_DIR', '/kaggle/working/artifacts'))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Artifacts will be saved to: {ARTIFACT_DIR}")

# Try common paths where your uploaded files might exist
possible_paths = [
    Path('/kaggle/input/jobs.csv'),
    Path('/kaggle/input/jobs/jobs.csv'),
    Path('/kaggle/input/jobs (1).csv'),
    Path('/mnt/data/jobs.csv'),
    Path('/mnt/data/jobs (1).csv'),
    Path('./jobs.csv'),
    Path('./jobs (1).csv')
]

found = {}
for p in possible_paths:
    if p.exists():
        found[p.name] = str(p)

if not found:
    print("Could not auto-detect dataset files. Please upload jobs.csv and 'jobs (1).csv' to the Kaggle notebook or adjust paths.")
else:
    print("Detected files:")
    for k,v in found.items():
        print(k, '->', v)

# Point to paths (update manually if auto-detection fails)
PATH_JOB1 = found.get('jobs.csv', '/kaggle/input/fvfvsdfvsfvs/jobs.csv') 
PATH_JOB2 = found.get('jobs (1).csv', '/kaggle/input/fvfvsdfvsfvs/jobs (1).csv')



Artifacts will be saved to: /kaggle/working/artifacts
Could not auto-detect dataset files. Please upload jobs.csv and 'jobs (1).csv' to the Kaggle notebook or adjust paths.


In [3]:
# ============================================================================
# Stage 1: Inspect datasets (first look + column types + null counts)
# ============================================================================
import pandas as pd

def load_csv_safe(path):
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"Failed to read {path}: {e}")
        return None

print('\nLoading datasets...')
df1 = load_csv_safe(PATH_JOB1)
df2 = load_csv_safe(PATH_JOB2)

print('\nDataset 1 (jobs.csv):')
if df1 is not None:
    display(df1.head(15))
    print('\nSchema and null counts:')
    display(pd.DataFrame({'dtype': df1.dtypes, 'nulls': df1.isna().sum(), 'unique': df1.nunique()}))

print('\nDataset 2 (jobs (1).csv):')
if df2 is not None:
    display(df2.head(15))
    print('\nSchema and null counts:')
    display(pd.DataFrame({'dtype': df2.dtypes, 'nulls': df2.isna().sum(), 'unique': df2.nunique()}))

# Save initial inspection summary
inspec = {
    'df1_shape': tuple(df1.shape) if df1 is not None else None,
    'df2_shape': tuple(df2.shape) if df2 is not None else None,
    'df1_columns': df1.columns.tolist() if df1 is not None else [],
    'df2_columns': df2.columns.tolist() if df2 is not None else []
}
with open(ARTIFACT_DIR / 'initial_inspection.json', 'w', encoding='utf-8') as f:
    json.dump(inspec, f, ensure_ascii=False, indent=2)
print('\nSaved initial inspection to', ARTIFACT_DIR / 'initial_inspection.json')




Loading datasets...

Dataset 1 (jobs.csv):


,Job,Company,Salary,Location,Type,Experience,Education,Industry,Position,Description,Source,URL,Keyword,Requirement,Benefit
0,Business Analyst (Giải Pháp Nhân Sự Tiền Lương),Fpt Is,Negotiable,Hà Nội,Full-time,2 year,Bachelor's,['IT/Software'],Staff/Specialist,"thu thập, phân tích và làm rõ yêu cầu nghiệp v...",topcv.vn,https://www.topcv.vn/viec-lam/business-analyst...,Business analyst,"tốt nghiệp đại học, ngành kinh tế, tài chính h...",thu nhập cạnh tranh theo năng lực ứng viên. lư...
1,Devops Engineer,Hạ Tầng Cmc Telecom,20-40,Hà Nội,Full-time,3 year or more,Bachelor's,['IT/Software'],Staff/Specialist,− quản lý sự cố / sự cố / thay đổi / bảo mật /...,topcv.vn,https://www.topcv.vn/viec-lam/devops-engineer/...,Devops,− có ít nhất 3 năm kinh nghiệm vận hành hệ th...,"đãi ngộ (lương, thưởng, review lương): – mức l..."
2,"Java Engineer Spring/Spring boot, Junit - From...",Rakus Việt Nam,Negotiable,Hồ Chí Minh,Full-time,3 year or more,Other,['Other'],Staff/Specialist,cơ hội làm việc với product hệ thống lớn - car...,vn.joboko.com,https://vn.joboko.com/viec-lam-java-engineer-s...,Java,có 3 năm kinh nghiệm trở lên về kiến thức và đ...,tại sao bạn nên chọn rakus việt nam? 1). lương...
3,"Kế Toán Tổng Hợp - Nghỉ T7,CN",Winsler Innovations,0-20,Hà Nội,Full-time,2 year,College/Associate,"['HR/Admin', 'Finance/Accounting']",Staff/Specialist,"- hạch toán chi phí, doanh thu, giá thành cho ...",topcv.vn,https://www.topcv.vn/viec-lam/ke-toan-tong-hop...,Accountant,- tốt nghiệp cao đẳng chuyên ngành kế toán hoặ...,- mức lương cứng từ 13.000.000 vnđ - 16.000.00...
4,Trưởng Phòng Phân Tích & Khai Thác Dữ Liệu,Ngân Hàng Tmcp Tiên Phong - Tpbank,Negotiable,Hà Nội,Full-time,3 year or more,Other,['Other'],Manager/Head,"1. quản lý, định hướng hoạt động của đội ngũ p...",vn.joboko.com,https://vn.joboko.com/viec-lam-truong-phong-ph...,Data engineer,• tốt nghiệp đại học trở lên các ngành khoa họ...,"thưởng thu nhập cạnh tranh, thưởng theo hiệu q..."
5,Kế Toán Thuế Dịch Vụ Làm Cho Khách Hàng FDI - ...,Nova Corporate Solutions,0-20,Hà Nội,Full-time,3 year or more,Bachelor's,"['HR/Admin', 'Finance/Accounting']",Staff/Specialist,tư vấn cho khách hàng (chủ yếu là người nước n...,topcv.vn,https://www.topcv.vn/viec-lam/ke-toan-thue-dic...,Accountant,tốt nghiệp đại học chuyên ngành kế toán ưu tiê...,lương cơ bản: từ 15 triệu tăng lương 1 đến 2 l...
6,Kế Toán Tổng Hợp - Thu Nhập Từ 13-15 Triêu - L...,Caron Holdings,0-20,Hà Nội,Full-time,2 year,College/Associate,"['HR/Admin', 'Finance/Accounting']",Staff/Specialist,-kiểm tra chứng từ thu/chi/nhập/xuất => hạch t...,topcv.vn,https://www.topcv.vn/viec-lam/ke-toan-tong-hop...,Accountant,"nữ, sức khỏe tốt. trình độ: tốt nghiệp cao đẳn...",mức lương: thu nhập 13-15 triệu (thoả thuận th...
7,Thực Tập Sinh Ngành Công Nghệ Thông Tin,Tm & Dv Nina,Negotiable,Hồ Chí Minh,Full-time,0 year,Vocational/Certificate,['IT/Software'],Intern/Entry-level,"thu thập và phân tích yêu cầu từ khách hàng, n...",topcv.vn,https://www.topcv.vn/viec-lam/thuc-tap-sinh-ng...,Business analyst,không yêu cầu kinh nghiệm đang là sinh viên nă...,"mức lương cạnh tranh, phù hợp với năng lực và ..."
8,Trưởng Nhóm Marketing (Leader Marketing) - Thu...,Brushie Official,20-40,Hà Nội,Full-time,3 year or more,Vocational/Certificate,['Marketing/Media'],Team Lead/Supervisor,1. vận hành & tối ưu quảng cáo - thiết lập mục...,topcv.vn,https://www.topcv.vn/viec-lam/truong-nhom-mark...,Marketing,"- có kinh nghiệm trong digital marketing, ưu t...",chế độ lương thưởng minh bạch & hấp dẫn - thu ...
9,"Senior Java Developer (Finance, E-Wallet...)",Kinh Doanh F88,20-40,Hà Nội,Full-time,2 year,Bachelor's,['IT/Software'],Staff/Specialist,"- phân tích yêu cầu, thiết kế hệ thống (sẽ đượ...",careerlink.vn,https://www.careerlink.vn/tim-viec-lam/senior-...,Java,- tối thiểu 03 năm kinh nghiệm lập trình front...,"môi trường làm việc trẻ trung, văn hóa doanh n..."



Schema and null counts:


,dtype,nulls,unique
Job,object,0,769
Company,object,0,640
Salary,object,0,6
Location,object,0,21
Type,object,0,4
Experience,object,0,4
Education,object,0,7
Industry,object,0,29
Position,object,0,5
Description,object,0,883



Dataset 2 (jobs (1).csv):


,job_title,job_type,position_level,city,experience,skills,job_fields,salary,salary_min,salary_max,unit
0,trưởng phòng kinh doanh,nhân viên chính thức,"trưởng nhóm , giám sát",hồ chí minh,lên đến 1 năm,NaN,"kinh doanh, bán hàng, nội ngoại thất, xây dựng",15 tr - 50 tr vnd,15.0,50.0,vnd
1,nhân viên qc ngành cơ khí,nhân viên chính thức,nhân viên,hồ chí minh,trên 1 năm,"production planning staff, chuyên viên iso, th...","vận hành sản xuất, sản xuất, qc), quản lý chất...",8 tr - 11 tr vnd,8.0,11.0,vnd
2,trưởng phòng đấu thầu,nhân viên chính thức,"trưởng nhóm , giám sát",hồ chí minh,5 - 7 năm,"trưởng phòng xây dựng, trưởng phòng đấu thầu, ...","điện, xây dựng, điện tử, điện lạnh, dầu khí, đ...",20 tr - 30 tr vnd,20.0,30.0,vnd
3,home textile designer,nhân viên chính thức,nhân viên,hưng yên,5 - 15 năm,NaN,"dệt may, nghệ thuật, da giày, thiết kế, mỹ thu...","800 - 1,500 usd",20.0,37.5,usd
4,nhân viên kinh doanh,nhân viên chính thức,nhân viên,hồ chí minh,trên 1 năm,NaN,"tư vấn, bán sỉ, bán lẻ, kinh doanh, bán hàng",10 tr - 50 tr vnd,10.0,50.0,vnd
5,giám đốc kinh doanh khu vực kem dẻo thổ nhĩ kỳ...,nhân viên chính thức,quản lý,bình định,2 - 5 năm,"sales director, giám đốc kinh doanh khu vực ke...","thực phẩm & đồ uống, bán sỉ, bán lẻ, kinh doan...",15 tr - 40 tr vnd,15.0,40.0,vnd
6,chuyên viên kinh doanh bất động sản,nhân viên chính thức,nhân viên,hà nội,lên đến 1 năm,"bất động sản, chuyên viên kinh doanh, bđs, rea...",bất động sản,15 tr - 50 tr vnd,15.0,50.0,vnd
7,kỹ thuật viên ie,nhân viên chính thức,nhân viên,hà nam,chưa có kinh nghiệm,kỹ thuật viên ie tiếng trung,"vận hành sản xuất, sản xuất",10 tr - 18 tr vnd,10.0,18.0,vnd
8,kế toán tổng hợp,nhân viên chính thức,nhân viên,bến tre,3 - 5 năm,"kế toán tổng hợp, nhân viên kế toán tổng hợp, ...","kế toán, kiểm toán, xuất nhập khẩu, vận hành s...",15 tr - 20 tr vnd,15.0,20.0,vnd
9,quản lý nhãn hàng,nhân viên chính thức,"trưởng nhóm , giám sát",hà nội,trên 2 năm,quản lý nhãn hàng,"marketing, y tế, dược phẩm, thẩm mỹ, hóa mỹ ph...",17 tr - 22 tr vnd,17.0,22.0,vnd



Schema and null counts:


,dtype,nulls,unique
job_title,object,7,23576
job_type,object,0,27
position_level,object,0,26
city,object,32,466
experience,object,0,181
skills,object,11294,20484
job_fields,object,7625,7491
salary,object,0,2069
salary_min,float64,0,190
salary_max,float64,0,175



Saved initial inspection to /kaggle/working/artifacts/initial_inspection.json


In [4]:
# ============================================================================
# Stage 2: Cleaning functions (text normalization, salary parsing, experience parsing)
# ============================================================================
import unicodedata

# Text normalization for Vietnamese and mixed-language fields
def normalize_text(text):
    if pd.isna(text):
        return ''
    if not isinstance(text, str):
        text = str(text)
    # normalize unicode
    text = unicodedata.normalize('NFC', text)
    # remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # unify whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Remove bullet symbols / formatting artifacts
    text = re.sub(r"[•●▪·\-–—]+", " ", text)
    # fix common prefixes like 'Mô tả công việc:'
    text = re.sub(r'(?i)mo\s?tả\s+công\s+việc:?', ' ', text)
    text = re.sub(r'(?i)yeu\s+cau:?', ' ', text)
    text = re.sub(r'(?i)\bï»¿\b', '', text)
    # Remove repeated punctuations and newlines
    text = re.sub(r"(\s*\n\s*)+", " ", text)
    text = re.sub(r"[ ]{2,}", " ", text)

    # Remove repeated boilerplate labels
    text = re.sub(r"(m[oô] t[aả] c[oô]ng vi[eệ]c|y[eê]u c[aà]u [uư]ng vi[eệ]n|quy[eê]n l[oơ]i|th[oô]ng tin th[eê]m):?", " ", text, flags=re.I)

    return text

# Salary parsing: handle VND, USD, ranges, words like 'thương lượng'
CURRENCY_PATTERNS = {
    'vnd': re.compile(r'(?i)\bvnd\b|\bvnđ\b|đ|₫'),
    'usd': re.compile(r'(?i)usd|\$')
}

def parse_salary_field(s):
    """Return (min, max, unit) where unit is 'vnd' or 'usd' or None. If missing, return (None,None,None)."""
    if pd.isna(s):
        return (None, None, None)
    s = str(s)
    s = s.replace(',', '.')  # some datasets use '.' as thousand sep or decimal
    s = s.replace('–', '-')
    # common patterns: '15-20 triệu', '15 triệu - 20 triệu', 'USD 1,000 - 2,000'
    # extract numbers
    nums = re.findall(r'\d+[\.,]?\d*', s)
    unit = None
    if re.search(r'(?i)usd|\$', s):
        unit = 'usd'
    elif re.search(r'(?i)vnd|vnđ|triệu|triệu|triệu|đồng|d', s):
        unit = 'vnd'

    if not nums:
        return (None, None, unit)

    # convert to float
    nums_f = []
    for n in nums:
        try:
            nums_f.append(float(n))
        except:
            try:
                nums_f.append(float(n.replace('.', '')))
            except:
                pass
    if not nums_f:
        return (None, None, unit)

    if len(nums_f) == 1:
        return (nums_f[0], nums_f[0], unit)
    else:
        return (nums_f[0], nums_f[-1], unit)

# Utility: convert units like 'triệu' meaning million VND; many postings use 'triệu' as shorthand
def normalize_salary_to_vnd(s_min, s_max, unit):
    if s_min is None and s_max is None:
        return (None, None)
    if unit is None:
        # assume vnd and unit is in millions if typical values small (e.g., <1000)
        unit = 'vnd'
    if unit == 'usd':
        # set conversion rate (adjustable)
        USD_TO_VND = 25000
        return (s_min * USD_TO_VND if s_min else None, s_max * USD_TO_VND if s_max else None)
    elif unit == 'vnd':
        # if s_min seems like 'triệu' (i.e., typical 10-50), interpret as million VND
        def fix_val(v):
            if v is None:
                return None
            if v < 1000:  # treat as million
                return v * 1_000_000
            else:
                return v
        return (fix_val(s_min), fix_val(s_max))
    else:
        return (s_min, s_max)

# Experience parsing: extract years
def parse_experience(exp):
    if pd.isna(exp):
        return None
    s = str(exp)
    nums = re.findall(r'\d+', s)
    if not nums:
        # look for words like 'Senior' -> infer level
        if re.search(r'(?i)senior|lead|manager', s):
            return 5
        if re.search(r'(?i)junior|intern|student|thực tập', s):
            return 0
        return None
    # No experience
    if re.search(r"kh[oô]ng y[eê]u c[aà]u|no experience", s):
        return 0

    # Range
    m = re.search(r"(\d+)\s*-\s*(\d+)", s)
    if m:
        return (int(m.group(1)) + int(m.group(2))) // 2

    # "Tren X nam"
    m = re.search(r"tr[eê]n\s*(\d+)", s)
    if m:
        return int(m.group(1)) + 1

    # "Duoi X nam"
    m = re.search(r"d[uư][oơ]i\s*(\d+)", s)
    if m:
        return 0

    # Single number
    m = re.search(r"(\d+)", s)
    if m:
        return int(m.group(1))
    # take first numeric as years
    try:
        return int(nums[0])
    except:
        return None

# Education mapping
EDU_MAP = {
    "trung cấp": 1,
    "cao đẳng": 1,
    "đại học": 2,
    "đại học trở lên": 2,
    "cử nhân": 2,
    "cao học": 3,
    "thạc sĩ": 3,
    "tiến sĩ": 4,
    "master": 3,
    "bachelor": 2,
    "phd": 4,
    "none": 0,
    "not required": 0,
    "không yêu cầu": 0,
}

def map_education(e):
    if pd.isna(e):
        return None
    s = str(e).lower()
    for k,v in EDU_MAP.items():
        if k in s:
            return v
    return None

# Location normalization
LOCATION_ALIASES = {
    'hcm': ['hồ chí minh', 'tp.hcm', 'tphcm', 'ho chi minh city', 'hcmc', 'hcm'],
    'hn': ['hà nội', 'ha noi', 'hn']
}

def normalize_location(loc):
    if pd.isna(loc):
        return None
    s = str(loc).lower()
    s = re.sub(r'\s+', ' ', s).strip()
    for k,alts in LOCATION_ALIASES.items():
        for a in alts:
            if a in s:
                return k
    return s

# Skill extraction: simple keyword extraction using regex (improvable with RAKE/KeyBERT)
def extract_skills(text):
    if not text:
        return []
    # naive: split by commas, semicolons, newlines, or 'và' / 'and'
    parts = re.split(r'[;,\n\|/]|\bvà\b|\band\b', text)
    skills = [p.strip() for p in parts if 2 <= len(p.strip()) <= 80]
    # further split multi-skills in parts separated by spaces if they contain '/'
    return list(dict.fromkeys([s for s in skills if s]))


In [5]:
# ============================================================================
# Stage 3: Clean each dataset using the above functions
# ============================================================================

def clean_jobs_df(df, source_name='df'):
    df = df.copy()
    # normalize column names
    df.columns = [c.strip() for c in df.columns]

    # common mappings (best effort)
    colmap = {}
    # detect similar names
    lowcols = {c.lower(): c for c in df.columns}
    def find_col(possible):
        for p in possible:
            if p.lower() in lowcols:
                return lowcols[p.lower()]
        return None

    colmap['job_title'] = find_col(['Job', 'job', 'job_title', 'job title', 'jobTitle'])
    colmap['description'] = find_col(['Description', 'description', 'Mô tả', 'Mo ta'])
    colmap['requirement'] = find_col(['Requirement', 'requirement', 'Yêu cầu', 'Yeu cau', 'Requirement'])
    colmap['benefit'] = find_col(['Benefit', 'benefit', 'Quyền lợi', 'Benefit'])
    colmap['salary'] = find_col(['Salary', 'salary', 'Salary_range', 'SalaryRange'])
    colmap['salary_min'] = find_col(['salary_min', 'Salary_min'])
    colmap['salary_max'] = find_col(['salary_max', 'Salary_max'])
    colmap['unit'] = find_col(['unit', 'currency'])
    colmap['experience'] = find_col(['Experience', 'experience', 'Kinh nghiệm', 'Exp'])
    colmap['education'] = find_col(['Education', 'education', 'Yêu cầu học vấn'])
    colmap['skills'] = find_col(['skills', 'Skill', 'skill', 'Keyword', 'keywords'])
    colmap['location'] = find_col(['Location', 'location', 'City', 'city'])
    colmap['company'] = find_col(['Company', 'company', 'Employer'])
    colmap['industry'] = find_col(['Industry', 'industry', 'job_fields'])
    colmap['position_level'] = find_col(['Position', 'position', 'position_level', 'level'])

    # Create unified columns
    for k,v in colmap.items():
        df[k] = df[v] if v in df.columns else None

    # Normalize text fields
    for tcol in ['job_title', 'description', 'requirement', 'benefit', 'skills', 'company', 'industry']:
        if tcol in df.columns:
            df[tcol] = df[tcol].apply(normalize_text)

    # Parse salary
    def compute_salary(row):
        # priority: salary_min & salary_max & unit columns if present
        smin = row.get('salary_min')
        smax = row.get('salary_max')
        unit = row.get('unit')
        if pd.notna(smin) or pd.notna(smax):
            try:
                sminf = float(smin) if pd.notna(smin) else None
            except:
                sminf = None
            try:
                smaxf = float(smax) if pd.notna(smax) else None
            except:
                smaxf = None
            mins, maxs = normalize_salary_to_vnd(sminf, smaxf, str(unit).lower() if pd.notna(unit) else None)
            return pd.Series({'salary_min_vnd': mins, 'salary_max_vnd': maxs})
        # else try to parse from 'salary' column
        s = row.get('salary')
        if pd.isna(s):
            return pd.Series({'salary_min_vnd': None, 'salary_max_vnd': None})
        pmin, pmax, unit = parse_salary_field(s)
        mins, maxs = normalize_salary_to_vnd(pmin, pmax, unit)
        return pd.Series({'salary_min_vnd': mins, 'salary_max_vnd': maxs})

    sal = df.apply(compute_salary, axis=1)
    df['salary_min_vnd'] = sal['salary_min_vnd']
    df['salary_max_vnd'] = sal['salary_max_vnd']

    # Experience
    df['experience_years'] = df['experience'].apply(parse_experience)

    # Education
    df['education_level'] = df['education'].apply(map_education)

    # Location
    df['location_norm'] = df['location'].apply(normalize_location)

    # Extract skill lists
    df['skills_list'] = df['skills'].apply(extract_skills)

    # Derive salary_mean
    def mean_salary(r):
        a = r['salary_min_vnd']
        b = r['salary_max_vnd']
        if pd.notna(a) and pd.notna(b):
            return (a + b) / 2
        if pd.notna(a):
            return a
        if pd.notna(b):
            return b
        return None
    df['salary_mean_vnd'] = df.apply(mean_salary, axis=1)

    # Keep only relevant columns in a canonical order
    keep_cols = ['job_title','company','description','requirement','benefit','skills','skills_list',
                 'salary_min_vnd','salary_max_vnd','salary_mean_vnd','experience_years','education_level',
                 'location_norm','industry','position_level','job_type']
    for c in keep_cols:
        if c not in df.columns:
            df[c] = None

    return df[keep_cols]

print('\nCleaning datasets...')
df1_clean = clean_jobs_df(df1, 'df1') if df1 is not None else None
df2_clean = clean_jobs_df(df2, 'df2') if df2 is not None else None

# Save cleaned intermediate outputs
if df1_clean is not None:
    df1_clean.to_csv(ARTIFACT_DIR / 'jobs_df1_cleaned.csv', index=False)
    print('Saved cleaned df1 to', ARTIFACT_DIR / 'jobs_df1_cleaned.csv')
if df2_clean is not None:
    df2_clean.to_csv(ARTIFACT_DIR / 'jobs_df2_cleaned.csv', index=False)
    print('Saved cleaned df2 to', ARTIFACT_DIR / 'jobs_df2_cleaned.csv')




Cleaning datasets...
Saved cleaned df1 to /kaggle/working/artifacts/jobs_df1_cleaned.csv
Saved cleaned df2 to /kaggle/working/artifacts/jobs_df2_cleaned.csv


In [6]:
# ============================================================================
# Stage 4: Merge datasets into unified schema and deduplicate
# ============================================================================
print('\nMerging cleaned datasets...')
if df1_clean is not None and df2_clean is not None:
    df_combined = pd.concat([df1_clean, df2_clean], ignore_index=True)
elif df1_clean is not None:
    df_combined = df1_clean.copy()
elif df2_clean is not None:
    df_combined = df2_clean.copy()
else:
    raise RuntimeError('No cleaned dataframes available')

# Basic deduplication by job_title + company + salary_mean_vnd
before = len(df_combined)
# normalize title+company for dedupe
_df = df_combined.copy()
_df['dup_key'] = (_df['job_title'].fillna('') + '|' + _df['company'].fillna('') + '|' + _df['salary_mean_vnd'].fillna('').astype(str))
df_combined = _df.drop_duplicates(subset=['dup_key']).drop(columns=['dup_key'])
after = len(df_combined)
print(f"Rows before dedupe: {before}, after dedupe: {after}")

# Save combined
df_combined.to_csv(ARTIFACT_DIR / 'jobs_combined_cleaned.csv', index=False)
print('Saved combined cleaned dataset to', ARTIFACT_DIR / 'jobs_combined_cleaned.csv')




Merging cleaned datasets...
Rows before dedupe: 86368, after dedupe: 33630
Saved combined cleaned dataset to /kaggle/working/artifacts/jobs_combined_cleaned.csv


In [7]:
# ============================================================================
# Stage 5: Feature engineering - build input_text and prepare target
# ============================================================================
import numpy as np 
print('\nFeature engineering...')

def build_input_text(row):
    parts = []
    for c in ['job_title','description','requirement','benefit','industry','position_level']:
        if row.get(c):
            parts.append(str(row[c]))
    return ' \n '.join(parts)

df_combined['input_text'] = df_combined.apply(build_input_text, axis=1)
# target: salary_mean_vnd; optionally log transform

df_combined['salary_target'] = df_combined['salary_mean_vnd']
# apply log1p transform for regression stability
df_combined['salary_target_log1p'] = df_combined['salary_target'].apply(lambda x: np.log1p(x) if pd.notna(x) else None)

# Save
df_combined.to_csv(ARTIFACT_DIR / 'jobs_combined_featured.csv', index=False)
print('Saved featured dataset to', ARTIFACT_DIR / 'jobs_combined_featured.csv')

# Create train/val/test split (only rows with salary available)
labeled = df_combined[df_combined['salary_target'].notna()].copy()
print('Labeled rows for salary prediction:', len(labeled))
from sklearn.model_selection import train_test_split
train, temp = train_test_split(labeled, test_size=0.3, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)
print('Train/val/test sizes:', len(train), len(val), len(test))

train.to_csv(ARTIFACT_DIR / 'train.csv', index=False)
val.to_csv(ARTIFACT_DIR / 'val.csv', index=False)
test.to_csv(ARTIFACT_DIR / 'test.csv', index=False)
print('Saved train/val/test sets to artifacts')


Feature engineering...
Saved featured dataset to /kaggle/working/artifacts/jobs_combined_featured.csv
Labeled rows for salary prediction: 33234
Train/val/test sizes: 23263 4985 4986
Saved train/val/test sets to artifacts


In [8]:
# ============================================================================
# Stage 6–8: PhoBERT Fine-tuning for Salary Prediction (Regression)
# ============================================================================
import os
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import numpy as np
import pandas as pd
from transformers import PhobertTokenizer, RobertaModel, get_scheduler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tqdm import tqdm

# =============================
# Config
# =============================
MODEL_DIR = "/kaggle/input/vinai-phobert-base/vinai-phobert-base"
tokenizer = PhobertTokenizer.from_pretrained(MODEL_DIR)
base_model = RobertaModel.from_pretrained(MODEL_DIR)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5
MAX_LEN = 256

# =============================
# Load Data
# =============================
ARTIFACT_DIR = Path("/kaggle/working/artifacts")
train = pd.read_csv(ARTIFACT_DIR / "train.csv")
val = pd.read_csv(ARTIFACT_DIR / "val.csv")
test = pd.read_csv(ARTIFACT_DIR / "test.csv")
print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

# =============================
# Dataset class
# =============================
class JobDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256, with_labels=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.with_labels = with_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.loc[idx, "input_text"])
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.with_labels:
            label = self.df.loc[idx, "salary_target_log1p"]
            item["labels"] = torch.tensor(label, dtype=torch.float32)
        return item

train_ds = JobDataset(train, tokenizer)
val_ds = JobDataset(val, tokenizer)
test_ds = JobDataset(test, tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# =============================
# Model definition
# =============================
class PhoBERTRegressor(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.roberta = base_model
        hidden_size = self.roberta.config.hidden_size
        self.dropout = nn.Dropout(0.2)
        self.regressor = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]  # CLS token
        pooled = self.dropout(pooled)
        preds = self.regressor(pooled).squeeze(-1)

        loss = None
        if labels is not None:
            loss_fn = nn.MSELoss()
            loss = loss_fn(preds, labels)
        return {"loss": loss, "preds": preds}

model = PhoBERTRegressor(base_model).to(DEVICE)

# =============================
# Optimizer & Scheduler
# =============================
optimizer = AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=total_steps
)

# =============================
# Training + Validation Loop
# =============================
def evaluate(model, val_loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating", leave=False):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            y = batch["labels"].to(DEVICE)
            out = model(input_ids, attention_mask)
            preds.append(out["preds"].detach().cpu().numpy())
            labels.append(y.cpu().numpy())
    preds = np.concatenate(preds)
    labels = np.concatenate(labels)
    rmse = np.sqrt(mean_squared_error(labels, preds))
    mae = mean_absolute_error(labels, preds)
    return rmse, mae

best_rmse = float("inf")


2025-10-27 08:46:12.662961: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761554772.827549      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761554772.875337      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Train: 23263 | Val: 4985 | Test: 4986


In [9]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        
        out = model(input_ids, attention_mask, labels)
        loss = out["loss"]
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    
    rmse, mae = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}: Train Loss={total_loss/len(train_loader):.4f} | Val RMSE={rmse:.4f}, MAE={mae:.4f}")
    
    if rmse < best_rmse:
        best_rmse = rmse
        torch.save(model.state_dict(), ARTIFACT_DIR / "phobert_best.pt")
        print("✅ Saved best model.")

Epoch 1/3: 100%|██████████| 2908/2908 [19:50<00:00,  2.44it/s]


Epoch 1: Train Loss=29.5777 | Val RMSE=0.6562, MAE=0.4506
✅ Saved best model.


Epoch 2/3: 100%|██████████| 2908/2908 [20:02<00:00,  2.42it/s]


Epoch 2: Train Loss=0.5753 | Val RMSE=0.6749, MAE=0.4644


Epoch 3/3: 100%|██████████| 2908/2908 [20:02<00:00,  2.42it/s]
                                                             

Epoch 3: Train Loss=0.5591 | Val RMSE=0.6715, MAE=0.4607


In [10]:
# =============================
# Load best model and Test
# =============================
model.load_state_dict(torch.load(ARTIFACT_DIR / "phobert_best.pt"))
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        y = batch["labels"].to(DEVICE)
        out = model(input_ids, attention_mask)
        test_preds.append(out["preds"].detach().cpu().numpy())
        test_labels.append(y.cpu().numpy())

test_preds = np.concatenate(test_preds)
test_labels = np.concatenate(test_labels)

rmse = np.sqrt(mean_squared_error(test_labels, test_preds))
mae = mean_absolute_error(test_labels, test_preds)

print(f"\n📊 Final Test RMSE: {rmse:.4f} | MAE: {mae:.4f}")

# Optional: convert predictions back to VND
test_df = test.copy()
test_df["predicted_salary_log1p"] = test_preds
test_df["predicted_salary_vnd"] = np.expm1(test_df["predicted_salary_log1p"])
test_df.to_csv(ARTIFACT_DIR / "test_with_predictions.csv", index=False)
print("Saved predictions to:", ARTIFACT_DIR / "test_with_predictions.csv")

Testing: 100%|██████████| 624/624 [01:21<00:00,  7.66it/s]



📊 Final Test RMSE: 0.6603 | MAE: 0.4502
Saved predictions to: /kaggle/working/artifacts/test_with_predictions.csv
